In [ ]:
import os
import sys
import subprocess
import base64
import pickle
from email.mime.text import MIMEText

from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

# =========================
# AUTO INSTALL PACKAGES
# =========================
packages = [
    "google-api-python-client",
    "google-auth-httplib2",
    "google-auth-oauthlib"
]

for package in packages:
    try:
        __import__(package.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# =========================
# GMAIL SETUP
# =========================
SCOPES = ['https://www.googleapis.com/auth/gmail.send']


def authenticate_gmail():
    creds = None

    if os.path.exists("token.pickle"):
        with open("token.pickle", "rb") as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:

        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json",
                SCOPES
            )
            creds = flow.run_local_server(port=0)

        with open("token.pickle", "wb") as token:
            pickle.dump(creds, token)

    return build("gmail", "v1", credentials=creds)


def create_email(sender, to, subject, body):
    msg = MIMEText(body)
    msg["to"] = to
    msg["from"] = sender
    msg["subject"] = subject

    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()
    return {"raw": raw}


def send_email(service, message):
    return service.users().messages().send(
        userId="me",
        body=message
    ).execute()


# =========================
# USER INPUT SECTION
# =========================

print("📧 Gmail Sender Program")

sender_email = input("Enter your Gmail (sender): ")
receiver_email = input("Enter receiver email: ")
subject = input("Enter subject: ")
body = input("Enter email body: ")

# =========================
# SEND EMAIL
# =========================

service = authenticate_gmail()

email_msg = create_email(sender_email, receiver_email, subject, body)

result = send_email(service, email_msg)

print("\n✅ Email sent successfully!")
print("Message ID:", result["id"])

📧 Gmail Sender Program
